# Keras to PyTorch - Who is doing the work behind `fit()`?

You know the learning routine. The awkward part of changing frameworks is
working out which jobs the old one did quietly and which ones the new one
expects you to own.

Keep one project throughout: **Antarctic Field Guide** predicts a Palmer
penguin's species from bill length, bill depth, flipper length, and body
mass. Four measurements go in; three raw species scores come out.
Species is the checking answer, never an input clue.

| Familiar operation | The responsibility to inspect |
|---|---|
| Arrays passed to layers | Feature floats, integer class IDs, matching devices |
| `Dense` | The same weighted sums, a different stored orientation |
| `from_logits=True` | Raw scores entering cross-entropy |
| `GradientTape` / `fit` | Graph lifetime, gradient buffers, and explicit updates |
| `predict` | Layer mode, graph tracking, class choice, and probability inspection |
| Try another model | Same split/budget; honest use of held-out rows |

The [Palmer Penguins data](https://allisonhorst.github.io/palmerpenguins/)
are CC0 and bundled by `palmerpenguins`; no data download is needed.
The executable cells use PyTorch only. Keras snippets are reference
translations, not a two-runtime benchmark.

We'll run storage/gradient checks, train the classifier, inspect decisions,
and compare widths/features. Mini-batch loading, dropout, and checkpoints
stay in the required Stage 02 route. Start with
[Tensors and Autograd](../../02-tensors-and-autograd/README.md) if shapes
and `.grad` are unfamiliar.
[Handwritten theory](01-keras-to-pytorch-antarctic-field-guide-theory.md).


One useful distinction before setup: an integer species ID names a score
column; it doesn't measure a quantity of species. A **logit** is that
column's raw score. Cross-entropy checks how strongly the scores favor the
correct ID. Softmax converts scores into relative probabilities when we
want to inspect them; PyTorch's loss should receive the raw scores.


![Four penguin measurements enter a sixteen-feature PyTorch model with three raw species scores; cross-entropy checks a separate species target and controlled ablations reuse held-out rows as validation.](../../images/penguin-framework-contracts.png)

*The project keeps the same learning goals while making PyTorch's tensor, training, and evaluation contracts explicit.*

## Keep the framework change smaller than the task

The sensor labs used made-up rows to isolate mechanisms. Penguins now add
real units, incomplete records, and uneven class counts without adding
image or sequence axes. The loop's jobs haven't changed.

The next cell checks dependencies and stops with setup instructions if
one is missing. It won't install into a shared kernel. TensorFlow isn't
required: only the PyTorch examples execute.


In [ ]:
import sys
print(f"Kernel interpreter: {sys.executable}")

# Setup: verify dependencies; do not install into a possibly shared notebook kernel.
import importlib.util
import sys

# Fail explicitly if a required package is missing; setup owns installation.
for package, module in [("palmerpenguins", "palmerpenguins"), ("seaborn", "seaborn")]:
    if importlib.util.find_spec(module) is None:
        raise ModuleNotFoundError(f"Missing {package}. Run this folder\'s setup in the intended environment.")

print("Setup check complete. The next cell imports the PyTorch-only runtime.")

### Repeat the starting draws; say where the tensors live

Python, NumPy, and PyTorch own separate random generators. Seed all three
before this recipe, then print the selected device. That repeats the
tested setup, not every library version or accelerator's exact arithmetic.

`.to(device)` moves tensors; it doesn't decide their meaning or dtype.
Inputs and weights must be in compatible memory for their operation.
Next we'll inspect which field records survived before building a model.


In [ ]:
#  Imports, deterministic seeds, and device selection
import random

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
from palmerpenguins import load_penguins

SEED = 17

# Seed Python, NumPy, and PyTorch separately since each owns its own RNG
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Prefer a GPU if visible to PyTorch, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch: {torch.__version__}")
print(f"Device:  {device}")
print("-> Keras users: explicit three-library seeding replaces keras.utils.set_random_seed(SEED).")

### Keras reference — reproducibility and model device

```python
keras.utils.set_random_seed(SEED)
# TensorFlow usually selects an available GPU automatically.
```

PyTorch keeps both choices explicit: seed Python, NumPy, and PyTorch, then move the model and tensors to one `device`. A device is not merely a speed setting; tensors can participate in one operation only when they live in the same memory space. A CPU input and GPU model therefore raise a device-mismatch error instead of being copied silently. Keeping `.to(device)` visible makes that data movement part of the program's contract.

## Part 1 - Four clues per bird, one separate answer

Keep bill length, bill depth, flipper length, and body mass in that order.
Drop records missing a required measurement or species before splitting,
and print how many remain. This chooses complete cases; it doesn't solve
missing-data bias.

Sort species names before assigning IDs: Adelie, Chinstrap, Gentoo.
That fixes which name each output column represents. Feature arrays use
floating point; label arrays use integers.

The next block keeps the original table for inspection, checks that the
bundled source hasn't become unexpectedly small, and prints actual class
counts. **Predict:** Are those counts equal? Use the field data, not a
balanced story we would like it to tell.


In [ ]:
#  Data extraction — CC0 Palmer Penguins measurements packaged with palmerpenguins
FEATURE_COLUMNS = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

# Load the bundled CC0 Palmer Penguins measurements
penguins = load_penguins()

# Drop rows missing any feature or the label before splitting
complete_penguins = penguins.dropna(subset=FEATURE_COLUMNS + ["species"]).reset_index(drop=True)

# Collect the sorted, distinct species names to fix a stable class order
species_names = sorted(complete_penguins["species"].unique())

# Map each species name to a stable integer class ID
species_to_id = {species: index for index, species in enumerate(species_names)}

# Convert features to float32 and labels to int64, the NumPy dtypes the tensor step expects
X_raw = complete_penguins[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
y_raw = complete_penguins["species"].map(species_to_id).to_numpy(dtype=np.int64)

# Fail fast if the CC0 package ever ships a materially smaller dataset
if len(complete_penguins) < 300:
    raise RuntimeError(f"Expected at least 300 complete penguin records; found {len(complete_penguins)}.")

print("Source: Palmer Penguins (CC0), bundled by palmerpenguins")
print(f"Rows retained after missing-value removal: {len(complete_penguins)}")
print(f"Feature shape: {X_raw.shape}; labels shape: {y_raw.shape}")
print("Class balance:", complete_penguins["species"].value_counts().to_dict())
print("-> The target is species; all four input columns are numeric field measurements.")


### A shape audit won't reveal a units problem

The next two panels show species counts and raw feature ranges.
Bill length is in millimeters; body mass is in grams. Overlaying them on
one value axis is intentionally awkward: an unscaled number can be large
because of its unit, not because its evidence is more important.

Normalization will give each feature a training-fitted ruler. It doesn't
make grams and millimeters the same physical quantity or repair uneven
collection. Read the printed counts and labelled axes before interpreting
an accuracy score.


In [ ]:
#  Data health check — inspect the CC0 field-data class balance and feature scales
# Plot species class counts alongside two feature-value histograms to sanity-check the data
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=complete_penguins, x="species", ax=axes[0], color="#4C78A8")
axes[0].set(title="Species labels in the CC0 dataset", xlabel="Species", ylabel="Penguins")
axes[0].tick_params(axis="x", rotation=20)
axes[1].hist(X_raw[:, 0], bins=20, alpha=0.8, label="bill length (mm)")
axes[1].hist(X_raw[:, 3], bins=20, alpha=0.6, label="body mass (g)")
axes[1].set(title="Two physical measurement distributions", xlabel="Feature value", ylabel="Penguins")
axes[1].legend()
plt.tight_layout()
plt.show()

# Confirm the expected feature/label shape before moving on
assert X_raw.shape[1] == 4 and len(species_names) == 3
print("-> The project has three observed species and four numeric features per penguin.")

The field table survived the shape/count checks. Its large raw ranges
aren't model inputs yet: next fit the training rulers and turn the
measurement and species roles into different tensor dtypes.


## Part 2 - Keep addresses separate from measurements

The next block shuffles **within each species**, puts about 80% in training,
and retains the rest. The floor at each class cutoff means the exact row
counts come from the printed tensors.

Fit four means and scales on training birds only. Their `[1,4]` shape is
one row of rulers reused across all birds, with one ruler per measurement.
Validation doesn't get to redefine them.

```text
features  [N,4] float32  -> four measurements per bird
labels    [N]   long     -> one correct species ID per bird
scores    [N,3] float    -> three possible species per bird, later
```

There is no one-hot target conversion hidden in that setup.
The names `X_test`/`y_test` are historical: later ablations use their
scores to choose settings, so those rows serve as **validation**, not a
final untouched test.


In [ ]:
#  Deterministic stratified split and train-only normalization
indices = np.arange(len(y_raw))
rng = np.random.default_rng(SEED)
train_indices, test_indices = [], []

# Split each species independently so the 80/20 ratio holds within every class
for label in range(len(species_names)):
    label_indices = indices[y_raw == label].copy()
    rng.shuffle(label_indices)
    cutoff = int(len(label_indices) * 0.8)
    train_indices.extend(label_indices[:cutoff])
    test_indices.extend(label_indices[cutoff:])
train_indices = np.array(sorted(train_indices))
test_indices = np.array(sorted(test_indices))
X_train_raw, X_test_raw = X_raw[train_indices], X_raw[test_indices]
y_train_np, y_test_np = y_raw[train_indices], y_raw[test_indices]

# Train table (N, 4) -> mean/std (1, 4): keep a row axis so statistics
# broadcast by measurement column across train and held-out rows without leakage.
mean = X_train_raw.mean(axis=0, keepdims=True)
std = X_train_raw.std(axis=0, keepdims=True)

# Guard against divide-by-zero for any near-constant feature
std[std < 1e-6] = 1.0
X_train_np, X_test_np = (X_train_raw - mean) / std, (X_test_raw - mean) / std

# Convert to the PyTorch dtype/device contract: float32 features, long class IDs
X_train = torch.as_tensor(X_train_np, dtype=torch.float32, device=device)
X_test = torch.as_tensor(X_test_np, dtype=torch.float32, device=device)
y_train = torch.as_tensor(y_train_np, dtype=torch.long, device=device)
y_test = torch.as_tensor(y_test_np, dtype=torch.long, device=device)

# Confirm the dtype and shape contract holds before training
assert X_train.dtype == torch.float32 and y_train.dtype == torch.long
assert X_train.shape[1] == 4 and X_train.shape[0] == y_train.shape[0]
print(f"X_train: {tuple(X_train.shape)}, {X_train.dtype}")
print(f"y_train: {tuple(y_train.shape)}, {y_train.dtype}")
print("-> float features + long class IDs are the PyTorch contract for nn.Linear + CrossEntropyLoss.")

### Same weighted sums, different shelf orientation

For the penguin task, four input measurements will feed three species
scores. First use a smaller three-input, two-output probe so every
coefficient fits on the page. It is a storage microscope, not a different
penguin dataset.

Keras exposes a feature-by-output kernel; `nn.Linear` exposes
output-by-feature weights. **Predict:** Copy unchanged, transpose, or
give up on parity? The fixed-weight calculation below settles it.


### Work out one row before copying the matrix

For probe `[1,2,-1]`, the first output is
`1×0.20 + 2×0.35 + (-1)×(-0.25) + 0.05 = 1.20`.
The second is `1×(-0.10) + 2×0.40 + (-1)×0.15 - 0.20 = 0.35`.
These coefficients are exactly the ones in the next cell.

The written kernel is `[3,2]`; PyTorch's stored weight is `[2,3]`.
Transpose **at copying time**, not by reordering input rows. The two-output
bias remains `[2]`.

> **Visual to add — Two shelves, the same contributions**
> Show probe values 1, 2, -1 feeding two sum trays. Label their contributions
> 0.20, 0.70, 0.25 plus bias 0.05 to total 1.20; and -0.10, 0.80,
> -0.15 plus bias -0.20 to total 0.35. Beside the trays place the exact
> three-by-two kernel from the cell and its two-by-three transpose.
> Suggested caption: "Transpose storage, not the calculation."

The next code copies those fixed values and asserts equality of its two
PyTorch calculations. No TensorFlow runtime is executed.

The Keras reference is simply `keras.layers.Dense(2, use_bias=True)`:
the final activation stays absent so these two outputs remain raw scores.


In [ ]:
#  Dense-layer parity proof — same affine math, transposed weight storage
torch.manual_seed(SEED)
keras_style_kernel = torch.tensor([[0.20, -0.10], [0.35, 0.40], [-0.25, 0.15]])
keras_style_bias = torch.tensor([0.05, -0.20])
probe = torch.tensor([[1.0, 2.0, -1.0]])
linear = nn.Linear(3, 2)

# (3 input features, 2 outputs) -> (2 outputs, 3 features): transpose only
# the stored kernel so nn.Linear can compute the same feature combinations.
with torch.no_grad():
    linear.weight.copy_(keras_style_kernel.T)
    linear.bias.copy_(keras_style_bias)

# Probe (1 row, 3 features) @ kernel (3, 2) + bias (2,) -> (1, 2 scores).
# Bias is reused by row; each output combines the same three input measurements.
keras_math = probe @ keras_style_kernel + keras_style_bias
pytorch_math = linear(probe)

# Confirm both frameworks produce identical outputs despite the storage-layout difference
assert torch.allclose(keras_math, pytorch_math)
print(f"Keras kernel shape:    {tuple(keras_style_kernel.shape)} = (in, out)")
print(f"PyTorch weight shape:  {tuple(linear.weight.shape)} = (out, in)")
print(f"Parity check: {torch.allclose(keras_math, pytorch_math)}")
print("-> Transpose at the storage boundary; the forward computation is the same affine map.")

![Keras and PyTorch implement the same affine map using transposed weight-storage layouts: Keras multiplies a row input by a three-by-two kernel, while PyTorch multiplies it by the transpose of a two-by-three weight matrix.](images/1.png)

*The forward computation is identical; only the exposed weight-storage orientation changes.*

The storage check returned the same outputs from both orientations.
Keep that separate from the next question: where does PyTorch put feedback,
and which line actually changes a parameter?


### Translate `GradientTape` on a calculation you already know

Square an adjustable value of 3. The output is 9, but a tiny input change
changes the square at **6** times that change. `.grad` should hold 6,
not 9.

`requires_grad=True` makes this leaf participate in recording.
`backward()` follows the route and writes the sensitivity into the leaf's
gradient buffer. The next cell checks it explicitly. No optimizer runs,
so the adjustable value remains 3.


In [ ]:
#  Autograd warm-up — same derivative as GradientTape, explicit gradient storage
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()

# Confirm the analytic derivative (dy/dx = 2x = 6.0 at x=3) matches autograd's result
assert x.grad.item() == 6.0
print(f"y = x² at x={x.item():.1f} gives y={y.item():.1f}")
print(f"dy/dx stored in x.grad: {x.grad.item():.1f}")
print("-> requires_grad records the operations; backward() writes the derivative into .grad.")

### Feedback still needs someone to move the knob

Start position at -2 and aim for 5. The squared distance is 49.
Its gradient is `2×(-2-5) = -14`; the first rate-0.08 update gives
`-2 - 0.08×(-14) = -0.88`. A negative gradient leads to a move right.

The next block repeats that same rule 80 times, using `no_grad` for the
parameter change and clearing the gradient afterward. `trajectory`
stores **post-update** positions, so its first point is not the -2 start.
It checks the final distance from 5 and plots the recorded path.

This number-line probe is not a penguin feature or class probability.
It isolates the updater's job before every classifier parameter gets feedback.


In [ ]:
#  Gradient-descent warm-up — use autograd to minimize a tiny known parabola
position = torch.tensor(-2.0, requires_grad=True)
target = 5.0
trajectory = []

# Run gradient descent by hand: compute loss, backprop, step, then clear the gradient
for _ in range(80):
    loss = (position - target) ** 2
    loss.backward()
    with torch.no_grad():
        position -= 0.08 * position.grad
    position.grad.zero_()
    trajectory.append(position.item())

# Confirm the position converged to the known target
assert abs(position.item() - target) < 1e-4

# Plot the position trajectory against the known minimum
plt.figure(figsize=(7, 3))
plt.plot(trajectory, color="#E45756")
plt.axhline(target, linestyle="--", color="#54A24B", label="minimum")
plt.title("Autograd drives a scalar toward a known minimum")
plt.xlabel("Step")
plt.ylabel("Position")
plt.legend()
plt.tight_layout()
plt.show()
print(f"Position after 80 steps: {position.item():.4f} (target {target:.1f})")
print("-> The same gradient-descent loop will update every Antarctic Field Guide weight together.")

#### Watch one autograd graph live and retire

![Animation tracing a PyTorch forward pass, dynamic computation graph creation, backward gradient flow, parameter update, gradient clearing, and graph disposal](images/pytorch-autograd-graph-lifecycle.gif)

Each forward pass records the operations that connected the input, parameters, and loss for that particular step. "Records" does not mean copying every tensor into a second model; PyTorch links each result to the operation that produced it, giving `backward()` a route for applying the chain rule in reverse.

Keep two lifetimes separate. The **computation graph** is normally released after `backward()`, then rebuilt by the next forward pass from the updated parameters. The resulting **gradient values** remain in each leaf parameter's `.grad` field until you clear them. The update changes parameter values; `zero_()` clears stored gradients, not the graph.

## Part 3 - Scores are useful before they are probabilities

Return to the field guide: four measurements become sixteen hidden
features, ReLU adds a bend, and a last layer makes three species scores.
The input for four birds is `[4,4]`; the output is `[4,3]`.
The model stores **131 parameters**, not one set per bird:
`4×16 + 16 + 16×3 + 3`.

Those intermediate sixteen-feature rows are activations for the current
birds. The weights are reusable parameters, and their gradients will
be a third set of tensors with matching parameter shapes.

In Keras this uses a final `Dense(3)` with no activation and a loss
configured `from_logits=True`. PyTorch's `CrossEntropyLoss` already
expects raw logits, so the same no-softmax choice belongs here.


![Two-lane classifier contract: raw logits feed cross-entropy directly during training, while inference converts logits to probabilities before selecting a class; applying softmax before cross-entropy is marked incorrect.](images/2.png)

The next class registers the 4-to-16-to-3 layers and returns one raw score
row per bird. `X_train[:4]` keeps four examples, not four measurement
columns. No labels enter `forward`, and no optimizer step happens here.
The shape assertion checks four rows of three scores before training.


In [ ]:
#  Antarctic Field Guide model — the readable PyTorch equivalent of Keras Sequential
HIDDEN_WIDTH = 16


# Two-layer MLP: PyTorch's explicit analogue of a Keras Sequential classifier
class AntarcticFieldGuide(nn.Module):
    def __init__(self, input_features=4, hidden_width=HIDDEN_WIDTH, classes=3):
        super().__init__()
        # (B penguins, input_features=4) -> (B, hidden_width) -> (B, classes=3).
        # ReLU keeps the hidden shape but makes the two-layer map nonlinear.
        self.layers = nn.Sequential(
            nn.Linear(input_features, hidden_width),
            nn.ReLU(),
            nn.Linear(hidden_width, classes),
        )

    def forward(self, features):
        return self.layers(features)  # (B, 4) -> (B, 3): one raw score per species.


model = AntarcticFieldGuide().to(device)
logits = model(X_train[:4])

# Confirm the model emits one logit row per input example, one column per class
assert logits.shape == (4, 3)
print(model)
print(f"Logit shape: {tuple(logits.shape)}; row 0 raw scores: {logits[0].detach().cpu().numpy()}")
print("-> nn.Module.forward() is the PyTorch analogue of Keras Layer.call().")

#### Call the model, then inspect a deliberately wrong loss

`model(...)` invokes the module's call machinery around `forward`,
including registered hooks. It uses the existing object; it doesn't
construct a new classifier or start training. Prefer that call over
bypassing it with `model.forward(...)`.

Now give cross-entropy two fixed score rows, `[3,-1]` and `[-0.5,1]`,
with correct class IDs 0 and 1. This two-class probe is separate from
the three-species model.

Raw scores in the first row differ by 4. Softmax turns them into
probabilities; feeding those probabilities to cross-entropy makes it
interpret them as **new scores**, not as the already-normalized result.
Both calls have acceptable shapes, so no shape error will save you.

> **Visual to add — A four-point lead gets squeezed twice**
> Show the exact first score row `[3,-1]` with target 0 entering two lanes:
> raw scores straight to cross-entropy; scores through softmax first,
> then incorrectly treated as new logits. Mark the raw gap 4 and the
> probability gap "less than 1"; leave loss/gradient numbers for the
> actual printout rather than drawing invented values.
> Suggested caption: "The second softmax doesn't know the first one happened."

The next cell prints both losses and gradient norms. Follow the mistake
into feedback, not just into a different-looking probability row.


In [ ]:
#  Logits contract experiment — raw scores, not softmax probabilities, enter CrossEntropyLoss
fixed_logits = torch.tensor([[3.0, -1.0], [-0.5, 1.0]], requires_grad=True)
fixed_targets = torch.tensor([0, 1], dtype=torch.long)
loss_fn = nn.CrossEntropyLoss()

# Correct usage: feed raw logits directly into CrossEntropyLoss
correct_loss = loss_fn(fixed_logits, fixed_targets)

# Softmax keeps (2 examples, 2 class scores) but changes scores to probabilities.
# Passing them as new logits makes CrossEntropyLoss normalize again: a semantic error,
# not a shape error.
incorrect_loss = loss_fn(torch.softmax(fixed_logits, dim=1), fixed_targets)

# Compare gradient magnitudes to show the "incorrect" call still runs but distorts gradients
correct_grad = torch.autograd.grad(correct_loss, fixed_logits, retain_graph=True)[0].norm().item()
incorrect_grad = torch.autograd.grad(incorrect_loss, fixed_logits)[0].norm().item()
print(f"CrossEntropyLoss(raw logits):      {correct_loss.item():.5f}; gradient norm {correct_grad:.5f}")
print(f"CrossEntropyLoss(softmax(logits)): {incorrect_loss.item():.5f}; gradient norm {incorrect_grad:.5f}")
print("-> Both calls run, but only raw logits honor the intended CrossEntropyLoss contract.")

The two-class probe shows why an acceptable tensor shape can still ask
the wrong question. For the real three-species model, keep raw logits
for cross-entropy. For inspection, convert them to probabilities later.
Now we can expose the work hidden inside a high-level training call.


## Part 4 - Unpack `fit()` into five jobs

Keras's compile/fit pairing chooses the checking rule and optimizer, then
coordinates repeated practice. Here those owners remain separate:

```text
clear gradients -> make scores -> check labels -> backward -> step
```

The one-step cell uses a **separate** seeded classifier so it doesn't
steal an update from the main training budget. Adam receives registered
parameters; it keeps its own running gradient summaries outside `forward`.
The following helper will repeat this routine and return its evidence.


![Four-stage PyTorch training cycle: clear gradients, run the forward pass and compute loss, backpropagate into parameter gradients, then update parameters; a warning branch shows gradient accumulation when clearing is skipped.](images/3.png)

The next step prints loss before the update and the total gradient norm
after backward. That norm sums squared feedback over every parameter and
takes a square root. It describes the feedback; it isn't a measured
parameter delta. Adam's following step makes the actual update.


In [ ]:
#  One explicit PyTorch optimization step — the four operations behind fit()
torch.manual_seed(SEED)
step_model = AntarcticFieldGuide().to(device)
step_optimizer = torch.optim.Adam(step_model.parameters(), lr=0.02)
step_optimizer.zero_grad()
step_logits = step_model(X_train)
step_loss = loss_fn(step_logits, y_train)
step_loss.backward()

# Each parameter gradient -> squared scalar sum; sum parameters and take sqrt for one L2 norm.
gradient_norm = torch.sqrt(sum(parameter.grad.pow(2).sum() for parameter in step_model.parameters())).item()
step_optimizer.step()
print(f"Loss before update: {step_loss.item():.4f}")
print(f"Total gradient norm: {gradient_norm:.4f}")
print("-> zero_grad -> forward -> loss.backward -> step is GradientTape + apply_gradients made explicit.")

### Read the long block by what it returns

`train_classifier` creates a fresh seeded model and Adam, runs the
requested full-batch epochs, and returns three things:

- the trained model, whose parameters changed;
- an array of cross-entropy losses measured **before** each update;
- an array of combined parameter-gradient norms from backward.

One epoch is one full-batch update here. The norm is an inspection summary;
Adam uses the individual gradient entries, not that one scalar.
The `zero_grad` argument exists so we can remove clearing in the next
controlled comparison.

The first call uses 160 updates and plots its actual arrays. Smaller loss
means better fitting of training labels; neither a small loss nor a small
gradient norm establishes biological understanding.


In [ ]:
#  Full-batch training — fixed seed, fixed split, actual project measurements
# Trains one classifier and records loss/gradient history; zero_grad toggles accumulation for the ablation below
def train_classifier(hidden_width=HIDDEN_WIDTH, zero_grad=True, epochs=160):
    torch.manual_seed(SEED)
    candidate = AntarcticFieldGuide(hidden_width=hidden_width).to(device)
    optimizer = torch.optim.Adam(candidate.parameters(), lr=0.02)
    history, grad_history = [], []
    for _ in range(epochs):

        # Skipping zero_grad lets gradients accumulate across epochs, on purpose, for the ablation
        if zero_grad:
            optimizer.zero_grad()
        logits = candidate(X_train)
        loss = loss_fn(logits, y_train)
        loss.backward()
        # Parameter-gradient entries -> scalar squared sums -> global scalar L2 norm.
        total_grad = torch.sqrt(sum(p.grad.pow(2).sum() for p in candidate.parameters())).item()
        optimizer.step()
        history.append(loss.item())
        grad_history.append(total_grad)
    return candidate, np.array(history), np.array(grad_history)


model, loss_history, grad_history = train_classifier()

# Plot the training loss and gradient-norm curves side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(loss_history, color="#4C78A8")
axes[0].set(title="Antarctic Field Guide training loss", xlabel="Epoch", ylabel="Cross-entropy")
axes[1].plot(grad_history, color="#F58518")
axes[1].set(title="Gradient norm per epoch", xlabel="Epoch", ylabel="L2 norm")
plt.tight_layout()
plt.show()
print(f"Loss: {loss_history[0]:.3f} -> {loss_history[-1]:.3f}")
print("-> The curve comes from this project's real data and this explicit optimizer loop.")

### What happens if the notepad is never cleared?

Run **45 updates per model**, as the next cell specifies. Both models use
the same seed, training birds, width, Adam rate, and budget. Only clearing
changes.

Every forward pass makes a fresh graph. With `zero_grad=False`, old leaf
gradient values remain and the next backward adds to them. That's not
reusing a released graph; it's accumulating feedback from different model
states.

**Predict:** Will the paths match, differ, or immediately raise an error?
The paired loss and gradient plots expose the actual consequence.
Intentional accumulation over planned micro-batches can be useful; this
experiment keeps stale feedback across full updates instead.


In [ ]:
#  Controlled ablation — gradient accumulation changes the optimization trajectory
# Train two identical models, one clearing gradients each epoch, one letting them accumulate
clean_model, clean_loss, clean_grad = train_classifier(zero_grad=True, epochs=45)
accum_model, accum_loss, accum_grad = train_classifier(zero_grad=False, epochs=45)

# Plot loss and gradient-norm curves for both conditions side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(clean_loss, label="zero_grad each epoch", color="#4C78A8")
axes[0].plot(accum_loss, label="gradients accumulate", color="#E45756")
axes[0].set(title="Loss changes when gradients accumulate", xlabel="Epoch", ylabel="Loss")
axes[0].legend()
axes[1].plot(clean_grad, label="cleared", color="#4C78A8")
axes[1].plot(accum_grad, label="accumulated", color="#E45756")
axes[1].set(title="Gradient norm reveals the mechanism", xlabel="Epoch", ylabel="L2 norm")
axes[1].legend()
plt.tight_layout()
plt.show()
print(f"Final loss with zero_grad: {clean_loss[-1]:.3f}")
print(f"Final loss without it:     {accum_loss[-1]:.3f}")
print("-> PyTorch accumulates gradients by design; clearing them is a semantic part of each training step.")

The ablation returned two trained models and two pairs of recorded arrays.
Read both loss and gradient histories before judging what forgetting to
clear did in this run. A tidy final loss isn't a substitute for the
update rule you intended.

We still haven't checked held-out birds. Training and prediction share
the model, but they shouldn't share permission to change it.


## Part 5 - Use the learned scores without another update

Select `model.eval()` for prediction-time layer behavior and `no_grad()`
to avoid an unused backward graph. This classifier has no dropout or batch
normalization, so eval mode doesn't change its values today. `no_grad`
alone wouldn't turn off either layer if added later.

The next block computes `[N,3]` logits and probabilities, then chooses
one ID per bird with `argmax(dim=1)`, giving `[N]`. The largest raw score
already selects the same class as softmax; probability conversion is for
inspection, not required for choosing the label.

Accuracy averages correctness across birds. The confusion matrix retains
**actual species down rows** and **predicted species across columns**.
That shows which mistakes an average might conceal. These held-out birds
will also guide the later ablations, so treat their score as validation.


In [ ]:
#  Held-out evaluation — no gradient graph, probabilities only after the model output
model.eval()

# Disable autograd tracking for inference-only forward passes
with torch.no_grad():
    test_logits = model(X_test)
    # (N birds, 3 species) -> (N, 3): normalize each row for readable confidence.
    test_probabilities = torch.softmax(test_logits, dim=1)
    # (N, 3) -> (N,): choose one species per bird, not one bird per species.
    test_predictions = test_logits.argmax(dim=1)
# Held-out correctness flags (N,) -> scalar mean: average birds, not species columns.
test_accuracy = (test_predictions == y_test).float().mean().item()
confusion = torch.zeros(len(species_names), len(species_names), dtype=torch.int64)

# Tally actual-vs-predicted species pairs into a confusion matrix
for actual, predicted in zip(y_test.cpu(), test_predictions.cpu()):
    confusion[actual, predicted] += 1

# Render the confusion matrix as an annotated heatmap
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(confusion.numpy(), annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=species_names, yticklabels=species_names, ax=ax)
ax.set(title="Held-out penguin species predictions", xlabel="Predicted species", ylabel="Actual species")
plt.tight_layout()
plt.show()
print(f"Held-out accuracy: {test_accuracy:.1%}")
print("-> eval() + no_grad() is PyTorch's explicit inference contract.")

### Open a few actual decisions

The next cell traces the first eight held-out decisions back to original
row indices, with actual species, chosen species, and the probability
assigned to that choice.

That probability expresses the model's relative preference among three
scores. It hasn't been tested for calibration; a large value can still
be wrong. Inspect the records alongside the confusion matrix rather than
treating confidence as an extra correctness label.


In [ ]:
#  Prediction audit — inspect actual species decisions and confidence
print("Row | actual species | predicted species | confidence")

# Walk the first 8 held-out rows and print the model's decision and confidence for each
for local_index, source_index in enumerate(test_indices[:8]):
    actual = species_names[y_test[local_index].item()]
    predicted_id = test_predictions[local_index].item()
    predicted = species_names[predicted_id]
    confidence = test_probabilities[local_index, predicted_id].item()
    print(f"{source_index:3d} | {actual:13s} | {predicted:17s} | {confidence:.2f}")
print("-> A classifier supplies evidence, not a biological verdict; inspect failures before trusting it.")

Once these birds' scores help you choose width or input features, they are
**validation data**, whatever `X_test` is named. A final quality claim
would need another untouched split. This notebook keeps the useful
comparisons without pretending it ran that extra test.


## Part 6 - More hidden features, or more useful clues?

### Your turn - change only the hidden-width list

For each width in `[4,16,64]`, reuse the same split, seed recipe, Adam
settings, four input features, and 160-update budget. Changing width
changes parameter shapes, so it can't preserve identical weight arrays.

The next block returns width, last recorded training loss, and held-out
accuracy for each trial, then plots both metrics. A wider model may fit
practice better without improving validation decisions.
**Predict:** Must width 64 win? Let the measured bars answer.


In [ ]:
#  CHANGE this list, then compare capacity with the same project conditions
WIDTHS = [4, 16, 64]
width_results = []

# Train and evaluate one classifier per hidden width, holding every other setting fixed
for width in WIDTHS:
    candidate, candidate_loss, _ = train_classifier(hidden_width=width, epochs=160)
    candidate.eval()
    with torch.no_grad():
        # (N birds, 3 scores) -> (N,) IDs -> scalar correctness mean.
        accuracy = (candidate(X_test).argmax(dim=1) == y_test).float().mean().item()
    width_results.append((width, candidate_loss[-1], accuracy))

# Bar-chart final loss and held-out accuracy side by side across widths
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar([str(width) for width, _, _ in width_results], [loss for _, loss, _ in width_results], color="#72B7B2")
axes[0].set(title="Final training loss", xlabel="Hidden width", ylabel="Loss")
axes[1].bar([str(width) for width, _, _, in width_results], [accuracy for _, _, accuracy in width_results], color="#54A24B")
axes[1].set(title="Held-out accuracy", xlabel="Hidden width", ylabel="Accuracy", ylim=(0, 1))
plt.tight_layout()
plt.show()
for width, loss, accuracy in width_results:
    print(f"width={width:2d}: loss={loss:.3f}, held-out accuracy={accuracy:.1%}")
print("-> More capacity can lower training loss without guaranteeing a better held-out decision rule.")

The width loop produced three matched-budget comparisons, not a new final
test. Now change the **clues**, keeping hidden width 16 and 160 updates.

`train_with_features` slices either bill length/depth or all four
normalized measurement columns, creates a matching input layer, trains,
and returns held-out accuracy. Species labels and their order don't change.
The input-layer shape changes as part of this declared feature comparison.

**Predict:** Must flipper length and mass improve the score? Report the
two printed accuracies honestly, including a tie. These rows still guide
selection; they don't become untouched again because a new plot was made.


In [ ]:
#  Feature-set ablation — bill dimensions alone vs. all four measurements
# Trains a fresh classifier restricted to the given feature columns and returns held-out accuracy
def train_with_features(feature_columns):
    local_train, local_test = X_train[:, feature_columns], X_test[:, feature_columns]
    torch.manual_seed(SEED)
    # (N birds, selected columns) -> (N, 16 features) -> (N, 3 species scores).
    # Changing input width lets each trial read only its chosen measurements.
    candidate = nn.Sequential(nn.Linear(len(feature_columns), 16), nn.ReLU(), nn.Linear(16, 3)).to(device)
    optimizer = torch.optim.Adam(candidate.parameters(), lr=0.02)
    for _ in range(160):
        optimizer.zero_grad()
        loss = loss_fn(candidate(local_train), y_train)
        loss.backward()
        optimizer.step()
    candidate.eval()

    # Evaluate without building an autograd graph
    with torch.no_grad():
        # (N birds, 3 scores) -> (N,) IDs -> scalar mean accuracy for this feature set.
        return (candidate(local_test).argmax(dim=1) == y_test).float().mean().item()


# Compare a 2-feature model against the full 4-feature model, same training conditions
bill_accuracy = train_with_features([0, 1])
full_feature_accuracy = train_with_features(list(range(4)))

# Bar-chart the two conditions' held-out accuracy
plt.figure(figsize=(6, 4))
plt.bar(["Bill dimensions", "All measurements"], [bill_accuracy, full_feature_accuracy], color=["#B279A2", "#59A14F"])
plt.ylim(0, 1)
plt.ylabel("Held-out accuracy")
plt.title("Feature-set ablation: one changed variable")
plt.show()
print(f"Bill dimensions only:  {bill_accuracy:.1%}")
print(f"All four measurements: {full_feature_accuracy:.1%}")
print("-> The measured difference, not the story we hoped for, decides whether extra features earned their keep.")

## The short version

- Four physical measurements enter; one integer species ID checks each
  three-score row. Labels aren't input features or one-hot targets here.
- Dense math survived the framework move. The fixed `[1,2,-1]` probe
  checks transposed storage with outputs 1.20 and 0.35.
- A forward graph, gradient buffers, parameters, and Adam's memory
  have different lifetimes. Backward writes feedback; step changes weights.
- Cross-entropy wants raw logits. The fixed-score experiment follows an
  unnecessary softmax into both loss and gradient changes.
- The no-clear comparison uses 45 updates per model. Fresh graphs
  don't remove old leaf gradients.
- Eval mode selects behavior; no-grad selects tracking. The confusion
  matrix and individual birds reveal more than average accuracy.
- Width and feature trials use held-out evidence for choices. That makes
  it validation, not an independent final test.

| Coverage | Where we stopped |
|---|---|
| Built and measured | Bundled-data audit; stratified split; train-only scaling; tensor roles; fixed-weight parity; derivative/update probes; logits loss check; explicit training; clearing ablation; held-out decisions; width/feature trials |
| Explained | Keras reference translations; model/gradient/optimizer ownership; probability versus calibration |
| Later | TensorFlow runtime comparisons, final-test evaluation, mini-batches, saved-state recovery, and deployment aren't run in this branch |

**Next:** [Vision](../vision/README.md) adds spatial locality, or return to
[Sequence Memory](../../../03-language-model-foundations/01-sequence-memory/README.md)
for ordered history. Neither needs penguin weights from this kernel.
